# 🇹🇬 Pipeline d'entraînement — Agent Agronome Kabyè

**Objectif** : Préparer les données et créer un dataset d'entraînement pour fine-tuner un LLM sur la langue kabyè (Togo Nord) dans le domaine agricole.

---

## Plan du notebook

| Étape | Description |
|-------|-------------|
| **1. Setup** | Imports, chemins, configuration |
| **2. Extraction PDF** | Extraire le dictionnaire/calendrier kabyè |
| **3. Transcription Audio** | Transcrire la Bible kabyè avec Whisper |
| **4. Nettoyage données** | Normaliser et structurer le texte kabyè |
| **5. Dataset Q&A** | Générer des paires question-réponse avec Claude |
| **6. Format JSONL** | Préparer le dataset au format fine-tuning |
| **7. Validation** | Vérifier la qualité du dataset |
| **8. Export** | Exporter pour fine-tuning + indexation RAG |

## Étape 1 — Setup & Configuration

In [ ]:
# Installation des dépendances (run once)
import subprocess, sys

packages = [
    'pymupdf',
    'anthropic',
    'openai-whisper',
    'datasets',
    'pandas',
    'tqdm',
    'python-dotenv',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('✅ Dépendances installées')

In [ ]:
import os, json, re, uuid
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import fitz  # PyMuPDF
import anthropic

# ── Configuration ──────────────────────────────────────────────
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', 'sk-ant-...')  # remplace ou exporte la var
CLAUDE_MODEL      = 'claude-sonnet-4-6'

# Chemins des ressources
RESOURCES_DIR = Path('../data/raw')          # dossier où tu mets tes fichiers
OUTPUT_DIR    = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fichiers sources (à adapter selon tes fichiers réels)
PDF_DICT_PATH   = RESOURCES_DIR / 'dictionnaire_fr_kabye.pdf'
PDF_CAL_PATH    = RESOURCES_DIR / 'Calendrierkabye2023.pdf'   # déjà disponible
AUDIO_BIBLE_PATH = RESOURCES_DIR / 'bible_kabye.mp3'

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print('✅ Configuration OK')
print(f'   Resources dir : {RESOURCES_DIR}')
print(f'   Output dir    : {OUTPUT_DIR}')

---
## Étape 2 — Extraction PDF

Extraction du texte brut depuis :
- Le **Calendrier Kabyè 2023** (Académie Kabiyè)
- Le **Dictionnaire français-kabyè** (si disponible)

In [ ]:
def extract_pdf(path: Path) -> list[dict]:
    """Extrait le texte page par page d'un PDF."""
    doc = fitz.open(str(path))
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if text:
            pages.append({
                'source': path.name,
                'page': i + 1,
                'text': text
            })
    print(f'📄 {path.name}: {len(pages)} pages extraites')
    return pages


# ── Calendrier Kabyè 2023 ───────────────────────────────────────
cal_pages = []
if PDF_CAL_PATH.exists():
    cal_pages = extract_pdf(PDF_CAL_PATH)
    for p in cal_pages[:3]:
        print(f'\n--- Page {p["page"]} ---')
        print(p['text'][:300])
else:
    print(f'⚠️  Fichier non trouvé: {PDF_CAL_PATH}')
    print('   → Copie le PDF dans data/raw/')

# ── Dictionnaire FR-Kabyè ──────────────────────────────────────
dict_pages = []
if PDF_DICT_PATH.exists():
    dict_pages = extract_pdf(PDF_DICT_PATH)
else:
    print(f'ℹ️  Dictionnaire non trouvé: {PDF_DICT_PATH} (optionnel)')

In [ ]:
def chunk_pages(pages: list[dict], chunk_size: int = 200, overlap: int = 30) -> list[dict]:
    """Découpe les pages en chunks de mots."""
    chunks = []
    for page in pages:
        words = page['text'].split()
        i = 0
        while i < len(words):
            chunk_words = words[i:i + chunk_size]
            chunk_text = ' '.join(chunk_words).strip()
            if len(chunk_text) > 30:  # ignore les chunks trop courts
                chunks.append({
                    'id': str(uuid.uuid4()),
                    'source': page['source'],
                    'page': page['page'],
                    'text': chunk_text,
                    'type': 'calendar' if 'Calendrier' in page['source'] else 'dictionary'
                })
            i += chunk_size - overlap
    return chunks


all_chunks = chunk_pages(cal_pages) + chunk_pages(dict_pages)
print(f'✅ {len(all_chunks)} chunks créés au total')

# Aperçu
df_chunks = pd.DataFrame(all_chunks)
if not df_chunks.empty:
    print(df_chunks[['source', 'page', 'type']].value_counts().head(10))

---
## Étape 3 — Transcription Audio (Bible Kabyè)

Utilise **Whisper** pour transcrire l'audio kabyè en texte.  
⚠️ Long pour un fichier entier — prévoir 10-30 min selon la taille.

In [ ]:
import whisper

def transcribe_audio(path: Path, model_size: str = 'medium') -> dict:
    """
    Transcrit un fichier audio avec Whisper.
    model_size: 'tiny' (rapide), 'base', 'small', 'medium' (recommandé), 'large'
    """
    print(f'🎙️  Chargement modèle Whisper ({model_size})...')
    model = whisper.load_model(model_size)
    
    print(f'🎙️  Transcription de {path.name}...')
    result = model.transcribe(
        str(path),
        language=None,       # auto-détection (kabyè proche du français phonétiquement)
        task='transcribe',
        verbose=True,
        word_timestamps=True
    )
    print(f'✅ Transcription terminée — {len(result["text"])} caractères')
    return result


transcription = None

if AUDIO_BIBLE_PATH.exists():
    transcription = transcribe_audio(AUDIO_BIBLE_PATH, model_size='medium')
    
    # Sauvegarde intermédiaire
    out = OUTPUT_DIR / 'bible_kabye_transcription.json'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump({
            'text': transcription['text'],
            'segments': transcription.get('segments', [])
        }, f, ensure_ascii=False, indent=2)
    print(f'💾 Sauvegardé: {out}')
    print('\nAperçu:\n', transcription['text'][:500])
else:
    print(f'ℹ️  Audio non trouvé: {AUDIO_BIBLE_PATH}')
    print('   → Copie le fichier audio dans data/raw/ et relance cette cellule')

In [ ]:
# Charger une transcription déjà existante (si Whisper déjà fait)
trans_file = OUTPUT_DIR / 'bible_kabye_transcription.json'

audio_chunks = []
if trans_file.exists():
    with open(trans_file, encoding='utf-8') as f:
        data = json.load(f)
    
    # Chunker par segments Whisper (meilleur découpage naturel)
    segments = data.get('segments', [])
    if segments:
        # Grouper les segments par blocs de ~200 mots
        current_block = []
        current_words = 0
        for seg in segments:
            current_block.append(seg['text'].strip())
            current_words += len(seg['text'].split())
            if current_words >= 200:
                audio_chunks.append({
                    'id': str(uuid.uuid4()),
                    'source': 'bible_kabye_audio',
                    'text': ' '.join(current_block),
                    'type': 'bible'
                })
                current_block = []
                current_words = 0
        if current_block:
            audio_chunks.append({'id': str(uuid.uuid4()), 'source': 'bible_kabye_audio', 'text': ' '.join(current_block), 'type': 'bible'})
    else:
        audio_chunks = chunk_pages([{'source': 'bible_audio', 'page': 1, 'text': data['text']}])

    print(f'✅ {len(audio_chunks)} chunks audio chargés')
else:
    print('ℹ️  Pas de transcription disponible — lance la cellule Whisper ci-dessus')

---
## Étape 4 — Nettoyage & Normalisation des données Kabyè

In [ ]:
def clean_kabye_text(text: str) -> str:
    """Nettoyage minimal — ne pas trop agresser les caractères kabyè spéciaux."""
    # Supprimer lignes vides multiples
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Supprimer espaces multiples
    text = re.sub(r' {2,}', ' ', text)
    # Supprimer caractères de contrôle sauf newlines
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text)
    return text.strip()


# Combiner toutes les sources
all_data = all_chunks + audio_chunks

# Nettoyage
for item in all_data:
    item['text'] = clean_kabye_text(item['text'])

# Filtrer les chunks trop courts (< 50 chars)
all_data = [d for d in all_data if len(d['text']) >= 50]

df = pd.DataFrame(all_data)
print(f'✅ Dataset brut: {len(df)} chunks')
if not df.empty:
    print(df['type'].value_counts())
    print(f'\nLongueur moyenne: {df["text"].str.len().mean():.0f} chars')
    print(f'Longueur max    : {df["text"].str.len().max()} chars')

---
## Étape 5 — Génération de paires Q&A avec Claude

Pour chaque chunk de texte kabyè, on demande à Claude de générer des **questions-réponses** dans le domaine agricole.  
C'est le cœur du dataset de fine-tuning.

Format cible :
```json
{
  "instruction": "Question en kabyè ou français",
  "input": "",
  "output": "Réponse de l'agronome kabyè"
}
```

In [ ]:
QA_SYSTEM_PROMPT = """Tu es un expert de la langue kabyè (Togo) et de l'agriculture africaine.
À partir du texte kabyè fourni, génère des paires question-réponse pour entraîner un agent agronome kabyè.

Règles:
1. Génère 3 à 5 paires Q&A par chunk
2. Mix: certaines questions en kabyè, d'autres en français
3. Les réponses doivent être en kabyè (avec traduction française entre parenthèses)
4. Questions agricoles en priorité (cultures, saisons, marchés, techniques)
5. Format JSON strict — tableau d'objets {"question": ..., "reponse": ...}

Réponds UNIQUEMENT avec le JSON, sans explication."""


def generate_qa_pairs(chunk_text: str, max_retries: int = 2) -> list[dict]:
    """Génère des Q&A pour un chunk de texte kabyè."""
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=1024,
                system=QA_SYSTEM_PROMPT,
                messages=[{
                    'role': 'user',
                    'content': f'Texte kabyè:\n\n{chunk_text}'
                }]
            )
            text = response.content[0].text.strip()
            # Extraire le JSON
            match = re.search(r'\[.*\]', text, re.DOTALL)
            if match:
                return json.loads(match.group())
        except Exception as e:
            print(f'  ⚠️  Erreur attempt {attempt+1}: {e}')
    return []


# Test sur 1 chunk avant de lancer tout
if all_data:
    test_chunk = all_data[0]
    print('🧪 Test sur le premier chunk...')
    print('Texte:', test_chunk['text'][:200])
    
    pairs = generate_qa_pairs(test_chunk['text'])
    print(f'\n✅ {len(pairs)} paires générées:')
    for p in pairs:
        print(f'  Q: {p.get("question", "")}')
        print(f'  R: {p.get("reponse", "")[:100]}')
        print()
else:
    print('⚠️  Pas de données — charge les PDFs ou la transcription audio d\'abord')

In [ ]:
# ── Génération en batch (sur tous les chunks) ──────────────────
# ⚠️  Coûte des tokens API — commence par MAX_CHUNKS petit (ex: 20)
MAX_CHUNKS = 20   # ← augmente progressivement

dataset_raw = []
chunks_to_process = all_data[:MAX_CHUNKS]

print(f'🚀 Génération Q&A sur {len(chunks_to_process)} chunks...')

for chunk in tqdm(chunks_to_process):
    pairs = generate_qa_pairs(chunk['text'])
    for pair in pairs:
        if 'question' in pair and 'reponse' in pair:
            dataset_raw.append({
                'id': str(uuid.uuid4()),
                'instruction': pair['question'],
                'input': '',
                'output': pair['reponse'],
                'source_type': chunk.get('type', 'unknown'),
                'source_file': chunk.get('source', ''),
            })

print(f'\n✅ {len(dataset_raw)} paires Q&A générées')

# Sauvegarde intermédiaire
df_qa = pd.DataFrame(dataset_raw)
if not df_qa.empty:
    df_qa.to_csv(OUTPUT_DIR / 'dataset_qa_raw.csv', index=False)
    print(f'💾 Sauvegardé: {OUTPUT_DIR}/dataset_qa_raw.csv')
    df_qa.head(5)

---
## Étape 6 — Format JSONL pour fine-tuning

On produit deux formats :
- **Alpaca** (`instruction / input / output`) — compatible la plupart des frameworks
- **Claude Messages** (format conversations) — pour fine-tuning via API Anthropic

In [ ]:
KABYE_SYSTEM = """Ŋ yaa agronome Kabyè tɔtɔna. 
Tee cee kɛɛ Kabyè kiŋ nɛ Français.
Nte yaa expert ɖe tɛ pɛ agriculture togo nord: ignames, sorgho, maïs, coton, arachides.
Tee caa ñɔɔtʋ n Kabyè (avec traduction française si nécessaire)."""


def to_alpaca_format(row: dict) -> dict:
    """Format Alpaca standard."""
    return {
        'instruction': row['instruction'],
        'input': row.get('input', ''),
        'output': row['output'],
        'system': KABYE_SYSTEM
    }


def to_claude_messages_format(row: dict) -> dict:
    """Format conversations Claude API."""
    return {
        'system': KABYE_SYSTEM,
        'messages': [
            {'role': 'user', 'content': row['instruction']},
            {'role': 'assistant', 'content': row['output']}
        ]
    }


if dataset_raw:
    # ── Format Alpaca ─────────────────────────────────────────
    alpaca_data = [to_alpaca_format(r) for r in dataset_raw]
    alpaca_path = OUTPUT_DIR / 'kabye_agro_alpaca.jsonl'
    with open(alpaca_path, 'w', encoding='utf-8') as f:
        for item in alpaca_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'✅ Alpaca JSONL: {alpaca_path} ({len(alpaca_data)} exemples)')

    # ── Format Claude Messages ─────────────────────────────────
    claude_data = [to_claude_messages_format(r) for r in dataset_raw]
    claude_path = OUTPUT_DIR / 'kabye_agro_claude_messages.jsonl'
    with open(claude_path, 'w', encoding='utf-8') as f:
        for item in claude_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'✅ Claude JSONL: {claude_path} ({len(claude_data)} exemples)')
else:
    print('⚠️  Pas de dataset — lance la génération Q&A (étape 5) d\'abord')

In [ ]:
# ── Dataset de seed MANUEL (haute qualité) ──────────────────────
# Ces exemples écrits à la main sont les plus importants — ils "ancrent" le comportement.

MANUAL_SEEDS = [
    {
        'instruction': 'Comment planter les ignames en kabyè ?',
        'input': '',
        'output': 'ÑûmŸ tee laa wiye AGOZA (mai) kɔ ɔOMAŋ (avril). Tee kɛ tɛ pɛ tɔtɔ nɛ ñûmŸ wiye — ñûmŸ tee laa ɖe pote kɛɛ tɛ pɛ.\n\n(En français: On plante les ignames en AGOZA/mai ou ɔOMAŋ/avril. On prépare le sol et on met le tubercule-semence dans un poteau de terre.)'
    },
    {
        'instruction': 'Ñɔɔtʋ wiye tɔ Kabyè ?',
        'input': '',
        'output': 'Ñɔɔtʋ wiye tɔ Kabyè: KujukŸ (Dimanche), Hodo (Lundi), PiyŸ (Mardi), CùlŸ (Mercredi), SŸrŸkŸwŸŋ (Jeudi), KemeŋŸ (Vendredi), MŸzŸŋ (Samedi).\n\nLùmaŋza Kabyè tɔ ñɔɔtʋ 7 wiye.'
    },
    {
        'instruction': 'Quels sont les mois kabyè pour la saison des pluies ?',
        'input': '',
        'output': 'Saison des pluies / Agoza tɔ Kabyèland: ɔOMAŋ (Avril) → ALOMA (Octobre).\n\nMois principaux: AGOZA (Mai) = semailles. MòSòGúM (Juin) = entretien. HASòYAɔè (Juillet) = désherbage. KòYèNA (Août) = entretien. SALAò (Septembre) = début récoltes. ALOMA (Octobre) = grandes récoltes ignames.'
    },
    {
        'instruction': 'Où se tient le marché de Kara le lundi ?',
        'input': '',
        'output': 'Hodo (Lundi) tɔ Kozah: lùmaŋza tɔ Piyà-Càƒÿ-Lándà nɛ Fÿyûûdà.\n\n(En français: Le lundi/Hodo, les marchés dans la région Kozah se tiennent à Piyà, Càƒÿ, Lándà et Fÿyûûdà. Le marché de Kara lui-même est actif tous les jours.)'
    },
    {
        'instruction': 'Comment dit-on "récolte des ignames" en kabyè ?',
        'input': '',
        'output': 'ÑûmŸ taa wiye = récolte des ignames.\n\nLa récolte des ignames (ñûmŸ taa) se fait en ALOMA (Octobre) et KAMèò (Novembre) dans la région Kabyè (Kozah, Binah, Togo Nord).'
    },
    {
        'instruction': 'Quelles cultures pour la saison sèche en pays kabyè ?',
        'input': '',
        'output': 'Saison sèche / KüLAŋ → LAKòò (Janvier → Mars) tɔ Kabyèland:\n\nMaraîchage le long des rivières (Kara et tributaires): tomates, oignons, piments, gombo. Préparation des champs pour la prochaine saison. Stockage des récoltes (ignames, sorgho, maïs). Vente au marché de Kara.'
    },
    {
        'instruction': 'Fête Sarakawa en kabyè — kesako ?',
        'input': '',
        'output': 'Sarakawa nëaƒûû yýý týzûû wiye — La fête de Sarakawa est une commémoration importante du peuple Kabyè (Togo). Elle se tient en janvier (KüLAŋ) et célèbre la résistance et l\'identité kabyè. C\'est aussi une période de rassemblement familial et communautaire.'
    },
]

# Sauvegarder les seeds manuels
seed_path = OUTPUT_DIR / 'kabye_seeds_manual.jsonl'
with open(seed_path, 'w', encoding='utf-8') as f:
    for item in MANUAL_SEEDS:
        f.write(json.dumps({**to_alpaca_format(item), 'source': 'manual'}, ensure_ascii=False) + '\n')

print(f'✅ {len(MANUAL_SEEDS)} exemples seed manuels sauvegardés: {seed_path}')

---
## Étape 7 — Validation du dataset

On vérifie la qualité avant l'export final.

In [ ]:
def validate_dataset(jsonl_path: Path) -> dict:
    """Valide un fichier JSONL dataset."""
    items = []
    errors = []
    
    with open(jsonl_path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                obj = json.loads(line)
                items.append(obj)
                # Vérifications basiques
                if not obj.get('instruction'):
                    errors.append(f'Ligne {i}: instruction vide')
                if not obj.get('output'):
                    errors.append(f'Ligne {i}: output vide')
                if len(obj.get('output', '')) < 20:
                    errors.append(f'Ligne {i}: output trop court')
            except json.JSONDecodeError as e:
                errors.append(f'Ligne {i}: JSON invalide — {e}')
    
    stats = {
        'total': len(items),
        'errors': len(errors),
        'avg_instruction_len': sum(len(i.get('instruction', '')) for i in items) / max(len(items), 1),
        'avg_output_len': sum(len(i.get('output', '')) for i in items) / max(len(items), 1),
    }
    return stats, errors


for fname in ['kabye_agro_alpaca.jsonl', 'kabye_seeds_manual.jsonl']:
    p = OUTPUT_DIR / fname
    if p.exists():
        stats, errors = validate_dataset(p)
        print(f'\n📊 {fname}')
        print(f'   Total exemples : {stats["total"]}')
        print(f'   Erreurs        : {stats["errors"]}')
        print(f'   Moy. instruction : {stats["avg_instruction_len"]:.0f} chars')
        print(f'   Moy. output      : {stats["avg_output_len"]:.0f} chars')
        if errors:
            print('   ⚠️  Erreurs:', errors[:5])

---
## Étape 8 — Export final & Résumé

Fusion de toutes les sources en un dataset final propre.

In [ ]:
import shutil

FINAL_DATASET = OUTPUT_DIR / 'kabye_agro_FINAL.jsonl'

all_examples = []

# Sources à fusionner (dans l'ordre de priorité)
sources = [
    ('kabye_seeds_manual.jsonl', 'manual'),       # haute qualité
    ('kabye_agro_alpaca.jsonl', 'generated'),      # généré par Claude
]

for fname, src_type in sources:
    p = OUTPUT_DIR / fname
    if p.exists():
        with open(p, encoding='utf-8') as f:
            for line in f:
                obj = json.loads(line)
                obj['source'] = src_type
                all_examples.append(obj)
        print(f'✅ {fname}: chargé')

# Dédupliquer sur l'instruction
seen = set()
deduped = []
for ex in all_examples:
    key = ex.get('instruction', '').strip().lower()
    if key not in seen:
        seen.add(key)
        deduped.append(ex)

# Shuffle
import random
random.seed(42)
random.shuffle(deduped)

# Écriture du dataset final
with open(FINAL_DATASET, 'w', encoding='utf-8') as f:
    for ex in deduped:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print(f'\n🎉 Dataset FINAL: {FINAL_DATASET}')
print(f'   Total exemples uniques : {len(deduped)}')
print(f'   Par source:')
for src in set(e.get('source') for e in deduped):
    count = sum(1 for e in deduped if e.get('source') == src)
    print(f'     {src}: {count}')

---
## ✅ Résumé & Prochaines étapes

### Ce que ce notebook produit

| Fichier | Description |
|---------|-------------|
| `kabye_seeds_manual.jsonl` | Exemples de haute qualité écrits manuellement |
| `bible_kabye_transcription.json` | Transcription Whisper de la Bible audio |
| `kabye_agro_alpaca.jsonl` | Q&A générés par Claude (format Alpaca) |
| `kabye_agro_claude_messages.jsonl` | Q&A format conversations Claude |
| `kabye_agro_FINAL.jsonl` | **Dataset final fusionné — prêt à l'emploi** |

### Prochaines étapes

1. **Ajouter plus de ressources** → plus de PDFs, plus d'audio, glossaires
2. **Augmenter MAX_CHUNKS** → plus de paires Q&A générées
3. **Fine-tuning** → utiliser le JSONL sur :
   - Hugging Face `trl` (SFTTrainer) pour open-source LLM
   - OpenAI fine-tuning API (`gpt-3.5-turbo`)
   - Anthropic Constitutional AI (quand disponible)
4. **RAG enrichi** → indexer le dataset dans ChromaDB (backend existant)

```bash
# Pour indexer dans le backend RAG existant:
cd ../backend
python -m scripts.ingest --dict ../data/processed/kabye_agro_FINAL.jsonl
```